# 03 — Financial sentiment / tone classifier

| | |
|---|---|
| Base | `nlpaueb/sec-bert-base` (or `distilroberta-base`) — **not** `ProsusAI/finbert` |
| Data | `takala/financial_phrasebank`, `sentences_allagree` |
| Task | 3-class sequence classification |
| T4 | batch 32, max_len 128, fp16, 3–4 epochs, lr 2e-5 |

**Why not fine-tune finbert:** `ProsusAI/finbert` was already trained on
Financial PhraseBank. Fine-tuning it on the same dataset and reporting the score
measures memorisation, not quality (anti-pattern #8). The training script refuses
a finbert base unless you explicitly acknowledge it.

**The fair comparison instead:** our model fine-tuned from a base encoder on our
own stratified split, versus `finbert` evaluated on *our held-out split*. That is
a real result either way — including the way where finbert wins.

In [ ]:
# Confirm we actually have the T4 this recipe is written for.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Checkpoints MUST live somewhere that survives the VM (section 5.5).
# /content is ephemeral - it vanishes with the runtime, which is exactly the
# failure checkpointing exists to defend against.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/affa'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('checkpoints ->', DRIVE_ROOT)

In [ ]:
# Clone or update the repo, and verify it is current. Re-running this notebook
# from the top after a disconnect must not silently train an old revision.
import os, subprocess

REPO_URL = 'https://github.com/YOUR_USERNAME/agentic-financial-filing-analyst.git'
REPO_DIR = '/content/agentic-financial-filing-analyst'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)

local  = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
remote = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '@{u}'],
                        capture_output=True, text=True).stdout.strip()

if remote and local != remote:
    print(f'repo is BEHIND origin (local {local[:8]} != remote {remote[:8]})')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    print('pulled; RESTART THE RUNTIME so the new code is imported')
else:
    print(f'repo is current at {local[:8]}')

os.chdir(REPO_DIR)

In [ ]:
# datasets<4.0 is REQUIRED, not a preference: finer-139, financial_phrasebank
# and finqa are loading-script datasets, and datasets>=4.0 removed script
# execution entirely. The parquet mirrors are NOT equivalent - at least one is
# deduplicated, which changes the splits and breaks comparability.
%pip install -q -e ".[train,eval]"
%pip install -q "datasets>=2.19,<4.0"

import datasets, transformers
print('datasets', datasets.__version__, '| transformers', transformers.__version__)
assert int(datasets.__version__.split('.')[0]) < 4, (
    'datasets>=4.0 cannot execute loading scripts; pin datasets>=2.19,<4.0'
)

In [ ]:
SEED          = 42
BASE_MODEL    = 'nlpaueb/sec-bert-base'
PB_CONFIG     = 'sentences_allagree'   # also try sentences_75agree
MAX_LENGTH    = 128
BATCH_SIZE    = 32
EPOCHS        = 4
LR            = 2e-5
SAVE_STEPS    = 50                     # small dataset; steps come quickly
VAL_FRACTION  = 0.15
TEST_FRACTION = 0.15

CKPT_DIR = f'{DRIVE_ROOT}/sentiment'
print(CKPT_DIR)

## Checkpointing and resume

Colab runtimes disconnect, get recycled, and hit idle timeouts. Everything below
is built so a crash costs minutes, not the whole run.

**What resume restores:** model weights, optimizer moments, LR-scheduler
position, RNG state, global step, and dataloader position. That is why we resume
rather than "just train again from the saved weights" — restarting the optimizer
and the LR schedule from scratch is a *different run*, and its loss curve will
not join up with the first half.

**The cell below is idempotent.** Re-run it after a crash and it resumes
automatically, with no code edit.

**Determinism is a precondition.** `SEED`, `TRAIN_SAMPLES` and `EVAL_SAMPLES` are
written into the checkpoint directory as JSON, and the resume path *refuses* to
continue if they no longer match. Changing any of them after a crash means the
global step now points into different data and the resumed run is silently
meaningless (anti-pattern #14).

**Disk:** a full checkpoint is roughly 3–4× model size — fp32 weights plus two
AdamW moments — so `save_total_limit=2` is required, not tidiness, against
Drive's 15GB free tier. `save_steps` is set for ~15–20 minutes of training, not
per epoch: an epoch here is 40+ minutes and a disconnect at minute 39 loses all
of it.

In [ ]:
# Stratified split (PhraseBank is heavily skewed toward neutral), train/test
# duplicate check, checkpoint selection on validation.
!python training/train_sentiment.py \
    --output-dir "{CKPT_DIR}" \
    --base-model {BASE_MODEL} \
    --phrasebank-config {PB_CONFIG} \
    --seed {SEED} \
    --max-length {MAX_LENGTH} \
    --batch-size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --learning-rate {LR} \
    --save-steps {SAVE_STEPS} \
    --val-fraction {VAL_FRACTION} \
    --test-fraction {TEST_FRACTION} \
    --final-test

## Test the resume path — do not assume it

Untested resume logic is usually broken resume logic, and the moment you find
out is the moment you have already lost the run.

1. Run the training cell above and let it write at least two checkpoints.
2. **Runtime → Interrupt execution** (or just let the runtime die).
3. Re-run the training cell *unchanged*.

What you should see: `resuming from .../checkpoint-N`, and the loss continuing
from where it stopped rather than restarting near its initial value. If step
numbering restarts at 0, resume is not working — fix that before starting the
real run.

In [ ]:
# finbert scored on OUR held-out split, as the baseline. The output carries the
# caveat that finbert saw PhraseBank in training, so its number is an optimistic
# ceiling rather than a neutral reference.
!affa-eval sentiment \
    --model "{CKPT_DIR}/final" \
    --baseline ProsusAI/finbert \
    --phrasebank-config {PB_CONFIG} \
    --test-fraction {TEST_FRACTION} \
    --output eval_results/sentiment.json

import json
r = json.load(open('eval_results/sentiment.json'))
print('ours    ', r['metrics'])
print('finbert ', r['baseline_metrics'])
print('delta   ', r['deltas'])

### If your model loses to finbert

Record it and say so. A fine-tune that fails to beat its baseline stays in the
README with the measurement — that is a real finding about how much headroom
there is over a strong in-domain model, and deleting it would make every other
number in the repo less trustworthy.

In [ ]:
# Push the model and a card carrying the REAL numbers and the subset size.
# A model card with aspirational numbers is worse than no card.
from huggingface_hub import notebook_login
notebook_login()

HUB_ID = 'YOUR_USERNAME/affa-sentiment'

card = f"""---
license: apache-2.0
tags: [finance, sec-filings, affa]
---

# affa-sentiment

Fine-tuned for the Agentic Financial Filing Analyst.

3-class financial tone classifier, fine-tuned from a base encoder (not from finbert) on a stratified PhraseBank split.

## Measured results

Fill these in from the evaluation cell above. Report the **test** split score,
the **baseline measured on the same data with the same protocol**, and the
training subset size. Do not paste a number from a paper here.

| metric | this model | baseline | notes |
|---|---:|---:|---|
| (fill in) | | | |

- Training subset: `TRAIN_SAMPLES` (state the number actually used)
- Seed: `SEED`
- Checkpoint selected on: validation split
- Test split touched: once

## Not financial advice

Research and educational use only.
"""

import pathlib
pathlib.Path(f'{CKPT_DIR}/final/README.md').write_text(card, encoding='utf-8')
print('model card written; review it before pushing')